In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset

pd.set_option('display.max_rows',None)
df = pd.read_csv('ghs-2023-hhold-v1.csv')

df.duplicated().sum()

np.int64(0)

In [32]:
df[df.duplicated]

,uqnr,personnr,psu,prov,head_popgrp,head_sex,head_age,hsg_maind,hsg_walls,hsg_roof,...,disab_hh,sevdisab_hh,econact_hh,totmhinc,stratum,geotype,metro,metro_code,rotation,house_wgt


In [33]:
def create_target(row):
    wat = str(row.get('wat_drinkwat', ''))
    san = str(row.get('san_toil', ''))
    
    # Inadequate water
    inadequate_wat = any(wat.startswith(code) for code in ['6.', '7.', '8.', '10.', '11.', '13.', '15.', '16.'])
    
    # Inadequate sanitation
    inadequate_san = any(san.startswith(code) for code in ['6.', '7.', '8.', '9.', '11.', '12.'])
    
    return 1 if (inadequate_wat or inadequate_san) else 0

# 1. Create target Y (0 = Adequate, 1 = Water/Sanitation Deficit)
df['target'] = df.apply(create_target, axis=1)

In [34]:
# 1. Calculate raw counts and percentages
counts = df['target'].value_counts()
proportions = df['target'].value_counts(normalize=True) * 100

# 2. Print formatted results
print(f"Class 0 (Adequate Water & Sanitation Access): {counts[0]:,} households ({proportions[0]:.2f}%).")
print(f"Class 1 (Water/Sanitation Deficit):          {counts[1]:,} households ({proportions[1]:.2f}%).")

Class 0 (Adequate Water & Sanitation Access): 15,107 households (72.19%).
Class 1 (Water/Sanitation Deficit):          5,820 households (27.81%).


In [35]:
df.isnull().sum()

uqnr                   0
personnr               0
psu                    0
prov                   0
head_popgrp            0
head_sex               0
head_age               0
hsg_maind              0
hsg_walls              0
hsg_roof               0
hsg_floor              0
hsg_cond_wall          0
hsg_cond_roof          0
hsg_cond_floor         0
hsg_rm_tot             0
hsg_tenure             0
hsg_rent               0
hsg_rdp                0
hsg_subsidy            0
wat_drinkwat           0
wat_dist               0
wat_time               0
wat_pipe               0
wat_treat              0
wat_mun_source         0
wat_meter              0
wat_pay                0
wat_inte_12mth         0
wat_inte_2days         0
wat_inte_altsource     0
wat_inte_15days        0
wat_bottle             0
san_toil               0
san_share              0
san_location           0
san_distance           0
san_washfac            0
san_wash               0
eng_access             0
eng_mainelect          0


In [36]:
# Flag entries containing 'Unspecified' or survey error codes
unspecified_counts = df.apply(lambda col: col.astype(str).str.contains('99\.|Unspecified', case=False, regex=True).sum())

# View top columns with unspecified responses
unspecified_df = pd.DataFrame({'Unspecified_Count': unspecified_counts})
unspecified_df[unspecified_df['Unspecified_Count'] > 0].sort_values(by='Unspecified_Count', ascending=False)

,Unspecified_Count
eng_light_alt,1510
eng_cook_alt,1510
totmhinc,423
house_wgt,273
Q1ntLanguage,183
Q2Dwelling,183
fin_reqinc,146
lab_salary_hh,78
agr_agrino,45
fsd_worried,33


In [38]:
# 1. Target Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='Set2')
plt.title("Household Water/Sanitation Deficit (0=Adequate, 1=Deficit)")
plt.xlabel("Target Class")
plt.show()



/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/363064175.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='target', data=df, palette='Set2')
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/363064175.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [39]:
# 2. Deficit by Province (Geographic View)
plt.figure(figsize=(10, 5))
df.groupby('prov')['target'].mean().sort_values().plot(kind='barh', color='coral')
plt.title("Proportion of Households with Service Deficits by Province")
plt.xlabel("Vulnerability Rate")
plt.ylabel("Province")
plt.show()



/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1295807616.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [40]:
# 3. Deficit by Geography Type
plt.figure(figsize=(8, 5))
sns.barplot(x='geotype', y='target', data=df, ci=None, palette='muted')
plt.title("Service Deficit Rate by Geography Type")
plt.ylabel("Proportion Vulnerable")
plt.xlabel("Geographical Type")
plt.show()



/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/2168053866.py:3: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(x='geotype', y='target', data=df, ci=None, palette='muted')
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/2168053866.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='geotype', y='target', data=df, ci=None, palette='muted')
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/2168053866.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
# 4. Feature-Target Relationship 2 (Deficit by Dwelling Type) - REPLACED BOXPLOT
plt.figure(figsize=(10, 5))
sns.barplot(x='hsg_maind', y='target', data=df, errorbar=None, palette='pastel', edgecolor='black')
plt.title("Visual 4: Service Deficit Rate by Main Dwelling Type")
plt.xlabel("Main Dwelling Type")
plt.ylabel("Proportion Vulnerable (Deficit Rate)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/3285725254.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='hsg_maind', y='target', data=df, errorbar=None, palette='pastel', edgecolor='black')
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/3285725254.py:8: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/3285725254.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [42]:
# 5. Missingness / Data Quality - REPLACED HEATMAP
features_to_check = ['prov', 'geotype', 'hsg_maind', 'hholdsz', 'totmhinc', 'head_sex', 'target']
completeness = (1 - df[features_to_check].isnull().mean()) * 100

plt.figure(figsize=(8, 4))
completeness.plot(kind='bar', color='teal', edgecolor='black')
plt.title("Visual 5: Data Completeness Audit (Missingness Check)")
plt.xlabel("Selected Features")
plt.ylabel("Percentage of Valid (Non-Missing) Data (%)")
plt.ylim(0, 110)
plt.axhline(100, color='red', linestyle='--') # Line to show 100%
plt.tight_layout()
plt.show()

/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/3430970.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [43]:
#2.3 and 2.4
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

features = ['prov', 'geotype', 'hsg_maind', 'hholdsz', 'totmhinc', 'head_sex']
X = df[features]
y = df['target']

# 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

numeric_features = ['hholdsz', 'totmhinc']
categorical_features = ['prov', 'geotype', 'hsg_maind', 'head_sex']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [ ]:
!pip install tensorflow

In [ ]:
!pip install imbalanced-learn

In [44]:
# 1. This command fixes the "ModuleNotFoundError: No module named 'imblearn'"

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# (Assuming X_train, y_train, X_val, y_val, and preprocessor are already defined from Question 2)

# 1. Transparent Naïve Baseline
baseline = ImbPipeline([
    ('pre', preprocessor), 
    ('clf', DummyClassifier(strategy='stratified', random_state=42))
])

# 2. Traditional Model 1: Logistic Regression
lr_model = ImbPipeline([
    ('pre', preprocessor), 
    ('smote', SMOTE(random_state=42)), 
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# 3. Traditional Model 2: Random Forest
rf_model = ImbPipeline([
    ('pre', preprocessor), 
    ('smote', SMOTE(random_state=42)), 
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

# 4. Neural Model 1: Standard Scikit-Learn MLP
mlp_model = ImbPipeline([
    ('pre', preprocessor), 
    ('smote', SMOTE(random_state=42)), 
    ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', early_stopping=True, random_state=42))
])

# 5. Neural Model 2: Deep Scikit-Learn MLP (Alternative to TensorFlow)
# Uses a deeper architecture and L2 Regularization (alpha) to control overfitting
mlp_deep_model = ImbPipeline([
    ('pre', preprocessor), 
    ('smote', SMOTE(random_state=42)), 
    ('clf', MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu', alpha=0.01, early_stopping=True, max_iter=500, random_state=42))
])

print("Question 3 Models successfully built!")

Question 3 Models successfully built!


In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, average_precision_score, f1_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# 1. Load Data
df = pd.read_csv('ghs-2023-hhold-v1.csv')

# 2. Extract Numbers from the messy string columns
def extract_number(val):
    val = str(val)
    match = re.search(r'^(\d+)\.', val) # Look for "Number." at the start
    if match:
        num = float(match.group(1))
        # Handle 'Unspecified' income which is coded as 9999999
        return np.nan if num >= 9999999 else num
    
    digits = re.sub(r'\D', '', val) # Strip all non-digits
    return float(digits) if digits else np.nan

df['hholdsz_clean'] = df['hholdsz'].apply(extract_number)
df['totmhinc_clean'] = df['totmhinc'].apply(extract_number)

# 3. Create Target
def create_target(row):
    wat = str(row.get('wat_drinkwat', ''))
    san = str(row.get('san_toil', ''))
    inadequate_wat = any(wat.startswith(code) for code in ['6.', '7.', '8.', '10.', '11.', '13.', '15.', '16.'])
    inadequate_san = any(san.startswith(code) for code in ['6.', '7.', '8.', '9.', '11.', '12.'])
    return 1 if (inadequate_wat or inadequate_san) else 0

df['target'] = df.apply(create_target, axis=1)

# 4. Setup Features and Splits
features = ['prov', 'geotype', 'hsg_maind', 'hholdsz_clean', 'totmhinc_clean', 'head_sex']
X = df[features]
y = df['target']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

# 5. Preprocessing Pipeline (Using the newly cleaned columns)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), ['hholdsz_clean', 'totmhinc_clean']),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), ['prov', 'geotype', 'hsg_maind', 'head_sex'])
    ])

# 6. Define Models
models = {
    'Baseline': ImbPipeline([('pre', preprocessor), ('clf', DummyClassifier(strategy='stratified', random_state=42))]),
    'Logistic Regression': ImbPipeline([('pre', preprocessor), ('smote', SMOTE(random_state=42)), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))]),
    'Random Forest': ImbPipeline([('pre', preprocessor), ('smote', SMOTE(random_state=42)), ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))]),
    'Neural Net': ImbPipeline([('pre', preprocessor), ('smote', SMOTE(random_state=42)), ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', early_stopping=True, random_state=42))])
}

# 7. Evaluation Loop
results = []
plt.figure(figsize=(10, 8))

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_val)[:, 1]
    else:
        y_proba = y_pred 
        
    macro_f1 = f1_score(y_val, y_pred, average='macro')
    pr_auc = average_precision_score(y_val, y_proba)
    roc_auc = auc(*roc_curve(y_val, y_proba)[:2])
    
    results.append({'Model': name, 'Macro-F1': macro_f1, 'PR-AUC': pr_auc, 'ROC-AUC': roc_auc})
    
    print(f"\n================ {name} Validation Report ================")
    print(classification_report(y_val, y_pred))
    
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')
    
    # Plot smaller confusion matrices inline
    fig, ax = plt.subplots(figsize=(3, 2))
    sns.heatmap(confusion_matrix(y_val, y_pred), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    plt.title(f'{name} CM')
    plt.show()

# Finalize ROC
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.title('ROC Curves (Validation Set)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

# Finalize Comparison Chart
results_df = pd.DataFrame(results).melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(10, 5))
sns.barplot(x='Model', y='Score', hue='Metric', data=results_df, palette='viridis', edgecolor='black')
plt.title('Model Performance Comparison')
plt.ylim(0, 1.0)
plt.show()

/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1880904436.py:71: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 8))
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1880904436.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



================ Baseline Validation Report ================
              precision    recall  f1-score   support

           0       0.73      0.72      0.72      2266
           1       0.29      0.29      0.29       873

    accuracy                           0.60      3139
   macro avg       0.51      0.51      0.51      3139
weighted avg       0.60      0.60      0.60      3139


================ Logistic Regression Validation Report ================
              precision    recall  f1-score   support

           0       0.93      0.77      0.84      2266
           1       0.59      0.84      0.69       873

    accuracy                           0.79      3139
   macro avg       0.76      0.81      0.77      3139
weighted avg       0.83      0.79      0.80      3139



/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1880904436.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



================ Random Forest Validation Report ================
              precision    recall  f1-score   support

           0       0.86      0.79      0.83      2266
           1       0.55      0.67      0.61       873

    accuracy                           0.76      3139
   macro avg       0.71      0.73      0.72      3139
weighted avg       0.78      0.76      0.76      3139



/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1880904436.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



================ Neural Net Validation Report ================
              precision    recall  f1-score   support

           0       0.93      0.77      0.84      2266
           1       0.59      0.86      0.70       873

    accuracy                           0.79      3139
   macro avg       0.76      0.81      0.77      3139
weighted avg       0.84      0.79      0.80      3139



/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1880904436.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1880904436.py:106: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5p/h963d7dd0snb4rsg1x_s7ddh0000gn/T/ipykernel_8816/1880904436.py:114: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [46]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# 1. Ensure the preprocessor matches the EXACT cleaned columns created in Question 4
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), ['hholdsz_clean', 'totmhinc_clean']),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), ['prov', 'geotype', 'hsg_maind', 'head_sex'])
    ])

# 2. Re-instantiate the Provisional Champion (Random Forest) to ensure a clean slate
rf_model = ImbPipeline([
    ('pre', preprocessor), 
    ('smote', SMOTE(random_state=42)), 
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

# 3. Define the Search Space
param_grid = {
    'clf__max_depth': [10, 20, None],
    'clf__min_samples_split': [5, 10, 20],
    'clf__n_estimators': [100, 200]
}

# 4. Run the Grid Search strictly on the Training set
grid_search = GridSearchCV(rf_model, param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)

# 5. Lock in the Champion and evaluate ONE TIME on the completely unseen Test Set
final_champion_model = grid_search.best_estimator_
y_test_pred = final_champion_model.predict(X_test)

print("\nFinal Locked Pipeline Test Report:")
print(classification_report(y_test, y_test_pred))

Best Parameters: {'clf__max_depth': 20, 'clf__min_samples_split': 20, 'clf__n_estimators': 200}

Final Locked Pipeline Test Report:
              precision    recall  f1-score   support

           0       0.92      0.78      0.84      2267
           1       0.59      0.82      0.68       873

    accuracy                           0.79      3140
   macro avg       0.75      0.80      0.76      3140
weighted avg       0.83      0.79      0.80      3140



In [47]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Extract the final champion model directly from Question 5's GridSearch
final_model = grid_search.best_estimator_

# Re-instantiate the comparator (Logistic Regression) using the correct, 

lr_model = ImbPipeline([
    ('pre', preprocessor), 
    ('smote', SMOTE(random_state=42)), 
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_model.fit(X_train, y_train)

# ==========================================
# 6.1 ABLATION STUDY
# ==========================================
print("--- ABLATION STUDY RESULTS (Validation Set) ---")
baseline_f1 = f1_score(y_val, final_model.predict(X_val), average='macro')
print(f"Original Champion Macro-F1: {baseline_f1:.4f}")

# Ablation 1: Remove SMOTE (Imbalance Treatment)
rf_no_smote = ImbPipeline([
    ('pre', preprocessor), 
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])
rf_no_smote.fit(X_train, y_train)
print(f"Ablation 1 (No SMOTE) Macro-F1: {f1_score(y_val, rf_no_smote.predict(X_val), average='macro'):.4f}")

# Ablation 2: Remove Financial Input (totmhinc_clean)
features_no_fin = ['prov', 'geotype', 'hsg_maind', 'hholdsz_clean', 'head_sex']
X_train_no_fin, X_val_no_fin = X_train[features_no_fin], X_val[features_no_fin]

preprocessor_no_fin = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), ['hholdsz_clean']),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), ['prov', 'geotype', 'hsg_maind', 'head_sex'])
])
rf_no_fin = ImbPipeline([('pre', preprocessor_no_fin), ('smote', SMOTE(random_state=42)), ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))])
rf_no_fin.fit(X_train_no_fin, y_train)
# FIX: Evaluate using the original y_val
print(f"Ablation 2 (No Income Feature) Macro-F1: {f1_score(y_val, rf_no_fin.predict(X_val_no_fin), average='macro'):.4f}")

# Ablation 3: Remove Geography (prov, geotype)
features_no_geo = ['hsg_maind', 'hholdsz_clean', 'totmhinc_clean', 'head_sex']
X_train_no_geo, X_val_no_geo = X_train[features_no_geo], X_val[features_no_geo]

preprocessor_no_geo = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), ['hholdsz_clean', 'totmhinc_clean']),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), ['hsg_maind', 'head_sex'])
])
rf_no_geo = ImbPipeline([('pre', preprocessor_no_geo), ('smote', SMOTE(random_state=42)), ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))])
rf_no_geo.fit(X_train_no_geo, y_train)
# FIX: Evaluate using the original y_val
print(f"Ablation 3 (No Geography Features) Macro-F1: {f1_score(y_val, rf_no_geo.predict(X_val_no_geo), average='macro'):.4f}")


# 6.2 McNEMAR'S TEST

print("\n--- McNEMAR'S TEST (Test Set) ---")
# Predict on the locked TEST set
y_pred_rf = final_model.predict(X_test)
y_pred_lr = lr_model.predict(X_test)

# Calculate 2x2 contingency table segments
both_correct = sum((y_pred_rf == y_test) & (y_pred_lr == y_test))
rf_only = sum((y_pred_rf == y_test) & (y_pred_lr != y_test))
lr_only = sum((y_pred_rf != y_test) & (y_pred_lr == y_test))
both_wrong = sum((y_pred_rf != y_test) & (y_pred_lr != y_test))

table = [[both_correct, rf_only], 
         [lr_only, both_wrong]]

print("Contingency Table:")
print(f"Both Correct: {both_correct} | Champion Only: {rf_only}")
print(f"Comparator Only: {lr_only} | Both Wrong: {both_wrong}")

result = mcnemar([[table[0][1], table[1][0]], [table[0][0], table[1][1]]], exact=True)
print(f"\nMcNemar Exact p-value: {result.pvalue}")

--- ABLATION STUDY RESULTS (Validation Set) ---
Original Champion Macro-F1: 0.7716
Ablation 1 (No SMOTE) Macro-F1: 0.7157
Ablation 2 (No Income Feature) Macro-F1: 0.7668
Ablation 3 (No Geography Features) Macro-F1: 0.6118

--- McNEMAR'S TEST (Test Set) ---
Contingency Table:
Both Correct: 2379 | Champion Only: 106
Comparator Only: 62 | Both Wrong: 593

McNemar Exact p-value: 0.0


In [48]:
import joblib

joblib.dump(final_champion_model, 'locked_pipeline.pkl')
print("Model pipeline successfully saved as 'locked_pipeline.pkl'!")

Model pipeline successfully saved as 'locked_pipeline.pkl'!


In [ ]:
pip install streamlit joblib pandas

In [49]:
import streamlit as st
import joblib
import pandas as pd

# pipeline = joblib.load('locked_pipeline.pkl')

st.title("SA Municipal Water & Sanitation Prioritization Tool")
st.warning("Ethics Notice: This is a decision-support tool to prioritize proactive infrastructure audits. It must not be used to automate service denials.")

with st.form("household_data"):
    prov = st.selectbox("Province", ["1. Western Cape", "2. Eastern Cape", "7. Gauteng", "9. Limpopo"]) 
    geotype = st.selectbox("Geography Type", ["1. Urban", "2. Traditional", "3. Farms"])
    hsg_maind = st.selectbox("Dwelling Type", ["1. Formal dwelling", "2. Informal dwelling"])
    head_sex = st.selectbox("Head of Household Sex", ["1. Male", "2. Female"])
    hholdsz = st.number_input("Household Size", min_value=1, max_value=20, value=4)
    totmhinc = st.number_input("Total Monthly Income (ZAR)", min_value=0, value=2500)
    submitted = st.form_submit_button("Assess Household Risk")

if submitted:
    input_df = pd.DataFrame({'prov': [prov], 'geotype': [geotype], 'hsg_maind': [hsg_maind],
                             'hholdsz': [hholdsz], 'totmhinc': [totmhinc], 'head_sex': [head_sex]})

2026-10-20 23:40:47.992 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 23:40:48.002 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 23:40:48.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 23:40:48.009 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 23:40:48.012 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 23:40:48.017 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 23:40:48.019 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 23:40:48.025 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
import streamlit as st
import joblib
import pandas as pd

# Load the saved pipeline securely
@st.cache_resource
def load_pipeline():
    return joblib.load('locked_pipeline.pkl')

pipeline = load_pipeline()

st.title("SA Municipal Water & Sanitation Prioritization Tool")
st.write("### Decision-Support System for Proactive Municipal Infrastructure Audits")

# Responsible-use notice (Question 7.3)
st.warning("⚠️ **Ethics & Responsible Use Notice:** This tool identifies household infrastructural vulnerability based on Statistics South Africa survey data[cite: 15]. It must be used strictly as a proactive resource allocator for municipal audits and can **never** be used to automate service denials or infringe on basic constitutional rights[cite: 15].")

# User Input Form (Question 7.1 & 7.2)
with st.form("prediction_form"):
    st.subheader("Enter Household Parameters")
    
    col1, col2 = st.columns(2)
    
    with col1:
        prov = st.selectbox("Province", [
            "1. Western Cape", "2. Eastern Cape", "3. Northern Cape", 
            "4. Free State", "5. KwaZulu-Natal", "6. North West", 
            "7. Gauteng", "8. Mpumalanga", "9. Limpopo"
        ])
        geotype = st.selectbox("Geography Type", ["1. Urban", "2. Traditional", "3. Farms"])
        hsg_maind = st.selectbox("Main Dwelling Type", [
            "1. Formal dwelling", "2. Informal dwelling", "3. Traditional dwelling"
        ])
        
    with col2:
        head_sex = st.selectbox("Head of Household Sex", ["1. Male", "2. Female"])
        hholdsz_clean = st.number_input("Household Size", min_value=1, max_value=30, value=4, step=1)
        totmhinc_clean = st.number_input("Total Monthly Household Income (ZAR)", min_value=0, value=3500, step=500)
    
    submitted = st.form_submit_button("Assess Household Vulnerability")

# Prediction Execution and Validation (Question 7.2 & 7.3)
if submitted:
    try:
        # Wrap input into DataFrame matching training feature columns
        input_data = pd.DataFrame({
            'prov': [prov],
            'geotype': [geotype],
            'hsg_maind': [hsg_maind],
            'hholdsz_clean': [float(hholdsz_clean)],
            'totmhinc_clean': [float(totmhinc_clean)],
            'head_sex': [head_sex]
        })
        
        # Run through saved pipeline
        prediction = pipeline.predict(input_data)[0]
        probabilities = pipeline.predict_proba(input_data)[0]
        confidence = probabilities[prediction] * 100
        
        st.divider()
        st.subheader("Assessment Results")
        
        if prediction == 1:
            st.error(f"🔴 **High Vulnerability / Service Deficit Detected** (Confidence: {confidence:.1f}%)\n\n"
                     f"**Plain-Language Explanation:** This household exhibits risk indicators consistent with non-piped water sources or unsafe sanitation facilities[cite: 15]. \n\n"
                     f"👉 **Actionable Recommendation:** Flag this household/ward for priority municipal infrastructure inspection and mobile water tanker dispatch[cite: 15].")
        else:
            st.success(f"🟢 **Adequate Access Expected** (Confidence: {confidence:.1f}%)\n\n"
                       f"**Plain-Language Explanation:** This household's profile indicates standard access to piped water and flush/VIP sanitation facilities[cite: 15]. \n\n"
                       f"👉 **Actionable Recommendation:** Low priority for immediate emergency intervention.")
                       
    except Exception as e:
        st.error(f"⚠️ **Error in processing input:** {e}. Please ensure all fields are valid.")